# 02 — Preprocessing

Filters the raw Netflix dataset to remove noise and produce a clean dataset ready for model training.

The filtering decisions come from `01_eda.ipynb`:
- **Long tail on movies:** most movies have very few ratings and can't be modeled reliably → drop movies with fewer than `MIN_MOVIE_RATINGS` ratings.
- **Cold start on users:** users with very few ratings don't have enough history for collaborative filtering to work → drop users with fewer than `MIN_USER_RATINGS` ratings.

**Requires:** `01_eda.ipynb` to have been run first (produces `data/raw/data.parquet`).

**Outputs:**
- `data/processed/processed_data.parquet` — filtered dataset for model training
- `artifacts/valid_users.pkl` / `artifacts/valid_movies.pkl` — IDs seen during training, used to filter unknowns at inference
- `artifacts/user2idx.pkl` / `artifacts/item2idx.pkl` — deterministic integer index mappings for NCF embedding layers

In [8]:
from pathlib import Path
import pickle
import pandas as pd

### Parameters

Thresholds are set based on EDA findings. Adjust here and re-run to experiment with different values.

In [9]:
ROOT                = Path('../')
RAW_DATA_PATH       = ROOT / 'data' / 'raw'
PROCESSED_DATA_PATH = ROOT / 'data' / 'processed'
ARTIFACTS_PATH      = ROOT / 'artifacts'

MIN_MOVIE_RATINGS = 50   # movies below this threshold are too sparse to model
MIN_USER_RATINGS  = 10   # users below this threshold suffer from cold start

## 1. Load data

Loads the concatenated raw dataset saved by `01_eda.ipynb`.

In [10]:
df = pd.read_parquet(RAW_DATA_PATH / 'data.parquet')
print(f"Shape: {df.shape}")
df.head()

Shape: (100480507, 4)


,movie_id,customer_id,rating,date
0,1,1488844,3.0,2005-09-06
1,1,822109,5.0,2005-05-13
2,1,885013,4.0,2005-10-19
3,1,30878,4.0,2005-12-26
4,1,823519,3.0,2004-05-03


## 2. Filter sparse users and movies

We apply two filters to reduce noise:
- **Movies:** drop any movie with fewer than `MIN_MOVIE_RATINGS` ratings — not enough signal to learn a reliable embedding.
- **Users:** drop any user with fewer than `MIN_USER_RATINGS` ratings — too few interactions to model preferences (cold start).

Note: the user filter runs **after** the movie filter, since removing movies can reduce some users below the threshold.

In [11]:
before = len(df)

movie_counts = df['movie_id'].value_counts()
valid_movies = set(movie_counts[movie_counts >= MIN_MOVIE_RATINGS].index)
df = df[df['movie_id'].isin(valid_movies)]
print(f"After movie filter: {len(df):,} rows | {len(valid_movies):,} movies kept")

# Re-compute after movie filter — some users may now fall below the threshold
user_counts = df['customer_id'].value_counts()
valid_users = set(user_counts[user_counts >= MIN_USER_RATINGS].index)
df = df[df['customer_id'].isin(valid_users)]
print(f"After user filter:  {len(df):,} rows | {len(valid_users):,} users kept")

print(f"\nTotal dropped: {before - len(df):,} rows ({(before - len(df)) / before:.1%})")

After movie filter: 100,478,440 rows | 17,713 movies kept
After user filter:  100,394,331 rows | 463,770 users kept

Total dropped: 86,176 rows (0.1%)


### Save artifacts

We save four artifacts:

**`valid_users.pkl` / `valid_movies.pkl`** — used at inference to filter unknown IDs before calling the model.

**`user2idx.pkl` / `item2idx.pkl`** — integer index mappings required by the NCF embedding layers. These are created here (not in training) for two reasons:
1. **Coverage:** mappings must cover all valid users/movies, not just a training sample. Building them inside the training loop from a 5% sample would leave the remaining 95% unmappable at inference.
2. **Determinism:** sorting guarantees the same input always produces the same index, making runs reproducible across environments and time.

In [12]:
ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)

# Valid ID sets — for inference-time filtering of unknown users/movies
with open(ARTIFACTS_PATH / 'valid_movies.pkl', 'wb') as f:
    pickle.dump(valid_movies, f)
with open(ARTIFACTS_PATH / 'valid_users.pkl', 'wb') as f:
    pickle.dump(valid_users, f)

# Index mappings — sorted for deterministic, reproducible indices across runs
user2idx = {user_id: idx for idx, user_id in enumerate(sorted(valid_users))}
item2idx = {movie_id: idx for idx, movie_id in enumerate(sorted(valid_movies))}

with open(ARTIFACTS_PATH / 'user2idx.pkl', 'wb') as f:
    pickle.dump(user2idx, f)
with open(ARTIFACTS_PATH / 'item2idx.pkl', 'wb') as f:
    pickle.dump(item2idx, f)

print(f"valid_movies.pkl  — {len(valid_movies):,} IDs")
print(f"valid_users.pkl   — {len(valid_users):,} IDs")
print(f"user2idx.pkl      — {len(user2idx):,} entries  (indices 0 → {len(user2idx)-1:,})")
print(f"item2idx.pkl      — {len(item2idx):,} entries  (indices 0 → {len(item2idx)-1:,})")

valid_movies.pkl  — 17,713 IDs
valid_users.pkl   — 463,770 IDs
user2idx.pkl      — 463,770 entries  (indices 0 → 463,769)
item2idx.pkl      — 17,713 entries  (indices 0 → 17,712)


## 3. Save processed data

Parquet preserves dtypes (including `datetime64` for `date`) and is significantly faster to read than CSV for large datasets.

In [13]:
output_path = PROCESSED_DATA_PATH / 'processed_data.parquet'
df.to_parquet(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Shape: {df.shape}")
print(f"Unique users:  {df['customer_id'].nunique():,}")
print(f"Unique movies: {df['movie_id'].nunique():,}")
print(f"Date range:    {df['date'].min()} → {df['date'].max()}")

Saved: ../data/processed/processed_data.parquet
Shape: (100394331, 4)
Unique users:  463,770
Unique movies: 17,713
Date range:    1999-11-11 → 2005-12-31
